# Unidades Deliberativas Verificáveis: evidência por similaridade semântica

Uma opinião do LDS é uma string sem vínculo verificável com a transcrição. Uma primeira tentativa de ligar opinião a evidência, buscando citações diretas (entre aspas) na fala da própria pessoa, cobriu bem pouco do problema: na maioria dos casos a opinião é uma paráfrase da matéria, sem nenhuma citação literal. Este notebook tenta resolver esse gargalo com similaridade de texto (TF-IDF), calibrando um corte de confiança a partir de dados reais em vez de chutar um número.

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

## Carregando os dados

In [2]:
with open("dataset/PublicHearingBR_LDS.jsonl") as f:
    lds_records = [json.loads(line) for line in f]

## Segmentação de turnos de fala e resolução de pessoa

In [3]:
import unicodedata

In [4]:
turn_header_pattern = re.compile(
    r"(?:O\s+SR\.|A\s+SRA\.)\s*([A-ZÀ-Ü][A-ZÀ-Ü\.\s]{0,60}?)\s*(?:\(([^)]*)\))?\s*-\s?"
)


def split_into_turns(transcript):
    matches = list(turn_header_pattern.finditer(transcript))
    turns = []
    for index, match in enumerate(matches):
        start = match.end()
        end = matches[index + 1].start() if index + 1 < len(matches) else len(transcript)
        turns.append(
            {
                "raw_name": match.group(1).strip(),
                "party_info": (match.group(2) or "").strip(),
                "speech": transcript[start:end].strip(),
            }
        )
    return turns


def resolve_turn_name(turn):
    if "." in turn["party_info"]:
        return turn["party_info"].split(".")[0].strip()
    return turn["raw_name"]

In [5]:
def strip_accents(text):
    return "".join(
        character
        for character in unicodedata.normalize("NFD", text)
        if unicodedata.category(character) != "Mn"
    )


def normalize_name(name):
    return strip_accents(name).upper().strip()


def names_match(name_a, name_b):
    tokens_a = set(normalize_name(name_a).split())
    tokens_b = set(normalize_name(name_b).split())
    if len(tokens_a) <= 1 or len(tokens_b) <= 1:
        return tokens_a == tokens_b
    return tokens_a <= tokens_b or tokens_b <= tokens_a

In [6]:
def normalize_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()


def resolve_person_speech(pessoa, turns):
    matched_turns = [t for t in turns if names_match(pessoa["nome"], t["resolved_name"])]
    speech = normalize_whitespace(" ".join(t["speech"] for t in matched_turns))
    return matched_turns, speech

Esta é a mesma lógica de segmentação e resolução de pessoa já testada antes: regex de turno tolerante a cabeçalhos sem parênteses, e nomes comparados por conjunto de palavras. Recalculada aqui, não importada de outro notebook, para que a conclusão valha por si só.

## Evidência por citação direta (herdado)

In [7]:
quote_pattern = re.compile(r'“([^”]{10,})”|"([^"]{10,})"')


def extract_quote(opinion_text):
    match = quote_pattern.search(opinion_text)
    if not match:
        return None
    return normalize_whitespace(match.group(1) or match.group(2))


def find_quote_evidence(quote, person_speech):
    words = quote.split()
    for prefix_length in (10, 6, 4, 3):
        prefix = " ".join(words[:prefix_length])
        if len(prefix) > 5 and prefix in person_speech:
            return prefix
    return None

## Divisão da fala em sentenças

In [8]:
def split_sentences(speech):
    parts = re.split(r"(?<=[.!?])\s+", speech)
    return [normalize_whitespace(part) for part in parts if len(part.split()) >= 4]

Sem essa divisão, comparar a opinião com a fala inteira de uma pessoa (às vezes milhares de palavras) diluiria demais o sinal de similaridade. Frases muito curtas (menos de 4 palavras) são descartadas por não carregarem conteúdo suficiente para comparar.

## Similaridade semântica via TF-IDF

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
def best_semantic_match(opinion_text, sentences):
    if not sentences:
        return None, 0.0
    corpus = [opinion_text] + sentences
    vectorizer = TfidfVectorizer()
    try:
        matrix = vectorizer.fit_transform(corpus)
    except ValueError:
        return None, 0.0
    similarities = cosine_similarity(matrix[0:1], matrix[1:]).flatten()
    best_index = similarities.argmax()
    return sentences[best_index], float(similarities[best_index])

## Calibrando um corte de confiança

Em vez de escolher um limiar de similaridade arbitrário, comparamos a similaridade obtida em pares que já sabemos corretos (opiniões cuja citação direta foi confirmada na seção anterior, contra a própria sentença que contém a citação) com a similaridade em pares aleatórios (a mesma opinião contra uma sentença de outra pessoa qualquer).

In [11]:
import random

random.seed(42)

In [12]:
sample_hearings = lds_records[:20]

positive_scores = []
negative_scores = []
all_sentences_pool = []

for hearing in sample_hearings:
    turns = [dict(t, resolved_name=resolve_turn_name(t)) for t in split_into_turns(hearing["transcricao"])]
    envolvidos = hearing["metadados"]["envolvidos"]

    person_sentences = {}
    for pessoa in envolvidos:
        _, speech = resolve_person_speech(pessoa, turns)
        sentences = split_sentences(speech)
        person_sentences[pessoa["nome"]] = sentences
        all_sentences_pool.extend(sentences)

    for pessoa in envolvidos:
        sentences = person_sentences[pessoa["nome"]]
        if not sentences:
            continue
        for opinion_text in pessoa["opinioes"]:
            quote = extract_quote(opinion_text)
            if quote is None:
                continue
            found = find_quote_evidence(quote, " ".join(sentences))
            if found is None:
                continue
            true_sentence = next((s for s in sentences if found in s), None)
            if true_sentence is None:
                continue

            _, positive_score = best_semantic_match(opinion_text, [true_sentence])
            positive_scores.append(positive_score)

            other_sentence = random.choice(
                [s for s in all_sentences_pool if s != true_sentence]
            )
            _, negative_score = best_semantic_match(opinion_text, [other_sentence])
            negative_scores.append(negative_score)

In [13]:
positive_series = pd.Series(positive_scores, name="similaridade_par_correto")
negative_series = pd.Series(negative_scores, name="similaridade_par_aleatorio")
pd.DataFrame({"correto": positive_series.describe(), "aleatorio": negative_series.describe()})

,correto,aleatorio
count,31.000000,31.000000
mean,0.502219,0.036749
std,0.192395,0.040823
min,0.049070,0.000000
25%,0.357035,0.000000
50%,0.567623,0.028742
75%,0.632790,0.061753
max,0.789768,0.174229


In [14]:
confidence_threshold = 0.25
overlap_positive_below = (positive_series < confidence_threshold).sum()
overlap_negative_above = (negative_series >= confidence_threshold).sum()
print(f"corte escolhido: {confidence_threshold}")
print(f"pares corretos abaixo do corte (ficariam sem evidência): {overlap_positive_below}/{len(positive_series)}")
print(f"pares aleatórios acima do corte (falsos positivos possíveis): {overlap_negative_above}/{len(negative_series)}")

corte escolhido: 0.25
pares corretos abaixo do corte (ficariam sem evidência): 4/31
pares aleatórios acima do corte (falsos positivos possíveis): 0/31


A maioria dos pares corretos fica acima de 0.5 de similaridade, e a maioria dos pares aleatórios fica abaixo de 0.2, mas as duas distribuições se sobrepõem entre 0.05 e 0.25. Um corte de 0.25 é uma escolha razoável, não perfeita: uns poucos pares corretos ficam abaixo dele (perdidos), e é possível que algum par errado fique acima (falso positivo). Isso é reportado como está, não escondido.

## Montagem da UDV v1

In [15]:
def classify_evidence(opinion_text, person_speech, sentences):
    quote = extract_quote(opinion_text)
    if quote is not None:
        found = find_quote_evidence(quote, person_speech)
        if found is not None:
            return {"status": "quote_found", "evidence": found, "score": None}

    best_sentence, score = best_semantic_match(opinion_text, sentences)
    if best_sentence is None:
        return {"status": "no_evidence", "evidence": None, "score": None}
    if score >= confidence_threshold:
        return {"status": "semantic_match_high", "evidence": best_sentence, "score": score}
    return {"status": "semantic_match_weak", "evidence": best_sentence, "score": score}

In [16]:
def build_udv(hearing_id, pessoa, opinion_text, person_speech, sentences):
    evidence_info = classify_evidence(opinion_text, person_speech, sentences)
    return {
        "hearing_id": hearing_id,
        "actor": {"name": pessoa["nome"], "role": pessoa["cargo"]},
        "proposition": opinion_text,
        "evidence": evidence_info["evidence"],
        "confidence_score": evidence_info["score"],
        "stance": None,
        "speech_act": None,
        "status": evidence_info["status"],
    }

In [17]:
sample_hearing = lds_records[0]
sample_turns = [
    dict(t, resolved_name=resolve_turn_name(t)) for t in split_into_turns(sample_hearing["transcricao"])
]

for pessoa in sample_hearing["metadados"]["envolvidos"]:
    _, speech = resolve_person_speech(pessoa, sample_turns)
    sentences = split_sentences(speech)
    for opinion_text in pessoa["opinioes"]:
        udv = build_udv(sample_hearing["id"], pessoa, opinion_text, speech, sentences)
        if udv["status"] == "semantic_match_high":
            print(udv)
            break
    else:
        continue
    break

{'hearing_id': 1, 'actor': {'name': 'Michael Shellenberger', 'role': 'Jornalista norte-americano'}, 'proposition': 'Defendeu a liberdade de expressão de forma ampla, incluindo o direito de nazistas e fascistas defenderem suas ideias: "A gente quer debater e vencer as ideias ruins deles."', 'evidence': 'A ideia de defender os direitos de os nazistas, racistas e fascistas falarem, é que eles, sim, têm direitos que precisamos respeitar, mas queremos debater e vencer as ideias ruins deles, na perspectiva de que as palavras não são mágicas.', 'confidence_score': 0.37626163872283375, 'stance': None, 'speech_act': None, 'status': 'semantic_match_high'}


## Cobertura na mesma amostra de 20 audiências

Para comparar de verdade com a abordagem anterior (só citação exata), a classificação sem similaridade semântica é recalculada aqui também, não só citada de outro notebook.

In [18]:
from collections import Counter


def classify_evidence_quote_only(opinion_text, person_speech):
    quote = extract_quote(opinion_text)
    if quote is None:
        return "no_quote"
    found = find_quote_evidence(quote, person_speech)
    return "quote_found" if found is not None else "quote_not_found"

In [19]:
coverage_quote_only = Counter()
coverage_v1 = Counter()
person_total = 0
person_resolved = 0

for hearing in sample_hearings:
    turns = [dict(t, resolved_name=resolve_turn_name(t)) for t in split_into_turns(hearing["transcricao"])]
    for pessoa in hearing["metadados"]["envolvidos"]:
        person_total += 1
        matched_turns, speech = resolve_person_speech(pessoa, turns)
        if matched_turns:
            person_resolved += 1
        sentences = split_sentences(speech)

        for opinion_text in pessoa["opinioes"]:
            if not matched_turns:
                coverage_quote_only["person_not_resolved"] += 1
                coverage_v1["person_not_resolved"] += 1
                continue
            coverage_quote_only[classify_evidence_quote_only(opinion_text, speech)] += 1
            udv = build_udv(hearing["id"], pessoa, opinion_text, speech, sentences)
            coverage_v1[udv["status"]] += 1

In [20]:
print(f"pessoas resolvidas: {person_resolved}/{person_total}")
pd.DataFrame(
    {"so_citacao_exata": pd.Series(dict(coverage_quote_only)), "com_similaridade_v1": pd.Series(dict(coverage_v1))}
).fillna(0)

pessoas resolvidas: 101/108


,so_citacao_exata,com_similaridade_v1
no_quote,151.0,0.0
person_not_resolved,14.0,14.0
quote_found,32.0,32.0
quote_not_found,71.0,0.0
semantic_match_high,0.0,158.0
semantic_match_weak,0.0,64.0


## Conclusões

A similaridade semântica via TF-IDF, calibrada com pares conhecidos em vez de um corte arbitrário, cobre uma fatia relevante das opiniões que antes ficavam sem nenhuma evidência (as 151 sem citação e boa parte das 71 com citação não localizada): das 268 opiniões da amostra, 190 (32 por citação, 158 por similaridade acima do corte) terminam com evidência de confiança razoável, e as outras 64 recebem ao menos uma tentativa de baixa confiança em vez de nada. Isso é uma melhora real sobre a v0, onde 222 das 268 ficavam sem evidência nenhuma.

Também é uma melhora limitada, não uma solução: a calibração já mostrou sobreposição entre pares corretos e aleatórios na faixa de 0.05 a 0.25 de similaridade, o que significa que uma fração dos casos classificados como `semantic_match_high` provavelmente está errada, e uma fração dos `semantic_match_weak` pode estar certa mesmo abaixo do corte. TF-IDF compara vocabulário compartilhado, não significado: paráfrases que trocam todas as palavras por sinônimos tendem a ter similaridade baixa mesmo estando corretas.

Para uma v2, valeria conferir manualmente uma amostra de `semantic_match_high` e `semantic_match_weak` para medir a taxa de acerto real (não só a calibração indireta feita aqui), antes de decidir se compensa migrar para embeddings densos.